# Modul 10: Automatisierung — Cron, Hooks & Events

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook baust du einen Cron-Parser, ein Event-Trigger-System und einen Kosten-Kalkulator.

🎯 **Lernziel:** Zeitgesteuerte und event-basierte Automatisierung verstehen und Kosten kalkulieren.

---

## Event-Driven Architecture

```
  TRIGGER          CONDITION         ACTION
  ───────          ─────────         ──────
  Cron (Zeit)  →   Bedingung?   →   Task ausführen
  Webhook      →   Filter?      →   Agent benachrichtigen
  File-Change  →   Pattern?     →   Verarbeiten
```

**Autonomie-Budget:** Automatisierte Tasks brauchen **engere Policies** als interaktive Sessions — weil kein Mensch zuschaut.

In [ ]:
# Cron-Parser: Versteht Cron-Ausdrücke

class CronSchedule:
    """Parst und erklaert Cron-Ausdruecke."""

    # Felder: Minute, Stunde, Tag, Monat, Wochentag
    FIELDS = ['minute', 'hour', 'day', 'month', 'weekday']
    WEEKDAYS = ['So', 'Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa']

    def __init__(self, expression):
        parts = expression.strip().split()
        if len(parts) != 5:
            raise ValueError(f'Cron braucht 5 Felder, bekommen: {len(parts)}')
        self.raw = expression
        self.fields = dict(zip(self.FIELDS, parts))

    def explain(self):
        """Erklaert den Cron-Ausdruck in Deutsch."""
        parts = []

        m = self.fields['minute']
        h = self.fields['hour']
        d = self.fields['day']
        mo = self.fields['month']
        wd = self.fields['weekday']

        # Zeit
        if m != '*' and h != '*':
            parts.append(f'um {h}:{m.zfill(2)} Uhr')
        elif h != '*':
            parts.append(f'jede Minute in Stunde {h}')
        elif m.startswith('*/'):
            parts.append(f'alle {m[2:]} Minuten')
        else:
            parts.append('jede Minute')

        # Tag
        if d != '*':
            parts.append(f'am {d}. des Monats')

        # Wochentag
        if wd != '*':
            try:
                day_name = self.WEEKDAYS[int(wd)]
                parts.append(f'jeden {day_name}')
            except (ValueError, IndexError):
                parts.append(f'Wochentag {wd}')

        # Monat
        if mo != '*':
            parts.append(f'im Monat {mo}')

        return ', '.join(parts)

    def runs_per_day(self):
        """Schaetzt wie oft der Cron pro Tag laeuft."""
        m = self.fields['minute']
        h = self.fields['hour']

        if m.startswith('*/'):
            interval = int(m[2:])
            hours = 1 if h != '*' else 24
            return (60 // interval) * hours
        elif m != '*' and h != '*':
            return 1
        elif h != '*':
            return 60  # Jede Minute einer Stunde
        else:
            return 1440  # Jede Minute

    def __repr__(self):
        return f'Cron("{self.raw}"): {self.explain()}'


# Beispiele parsen
crons = [
    ('0 8 * * *',    'Taeglicher Digest'),
    ('0 9 * * 1',    'Montags-Review'),
    ('*/30 * * * *',  'Alle 30 Min Health-Check'),
    ('0 0 1 * *',    'Monatlicher Report'),
    ('30 18 * * 5',  'Freitags-Wochenreview'),
]

print('=== Cron-Parser ===\n')
for expr, desc in crons:
    cron = CronSchedule(expr)
    print(f'{desc}:')
    print(f'  Ausdruck: {expr}')
    print(f'  Bedeutung: {cron.explain()}')
    print(f'  Laeufe/Tag: {cron.runs_per_day()}')
    print()

In [ ]:
# Event-Trigger-System + Kosten-Kalkulator

class EventTrigger:
    """Ein einzelner Event-Trigger."""
    def __init__(self, name, trigger_type, condition, action, model='haiku'):
        self.name = name
        self.trigger_type = trigger_type  # 'cron', 'webhook', 'file'
        self.condition = condition
        self.action = action
        self.model = model
        self.execution_count = 0

    def check(self, event):
        """Prueft ob der Trigger auf ein Event reagiert."""
        return self.condition(event)

    def execute(self, event):
        self.execution_count += 1
        result = self.action(event)
        print(f'  ⚡ Trigger "{self.name}": {result}')
        return result


class AutomationEngine:
    """Verwaltet alle Trigger und kalkuliert Kosten."""

    # Kosten pro Aufruf (geschaetzt, in Dollar)
    MODEL_COSTS = {
        'haiku': 0.001,
        'sonnet': 0.01,
        'opus': 0.05,
    }

    def __init__(self):
        self.triggers = []

    def add(self, trigger):
        self.triggers.append(trigger)
        print(f'  Trigger registriert: {trigger.name} ({trigger.trigger_type})')

    def process_event(self, event):
        """Verarbeitet ein Event durch alle Trigger."""
        print(f'\n📨 Event: {event}')
        for trigger in self.triggers:
            if trigger.check(event):
                trigger.execute(event)

    def cost_estimate(self, days=30):
        """Kalkuliert geschaetzte Kosten ueber N Tage."""
        print(f'\n💰 Kosten-Kalkulation ({days} Tage):')
        print(f'{"Trigger":<25} {"Modell":<8} {"Aufrufe/Tag":<12} {"$/Tag":<10} {"$/Monat"}')
        print('-' * 70)

        total_monthly = 0
        for trigger in self.triggers:
            if trigger.trigger_type == 'cron' and hasattr(trigger, 'cron'):
                daily = trigger.cron.runs_per_day()
            else:
                daily = trigger.execution_count or 1  # Schaetzung

            cost_per_call = self.MODEL_COSTS.get(trigger.model, 0.01)
            daily_cost = daily * cost_per_call
            monthly_cost = daily_cost * days
            total_monthly += monthly_cost

            print(f'{trigger.name:<25} {trigger.model:<8} {daily:<12} ${daily_cost:<9.3f} ${monthly_cost:.2f}')

        print(f'\n  GESAMT: ${total_monthly:.2f} / {days} Tage')
        return total_monthly


# === Demo ===
print('=== Automatisierungs-Engine ===\n')
engine = AutomationEngine()

# Cron-Trigger: Taeglicher Digest
digest_trigger = EventTrigger(
    'daily-digest', 'cron',
    condition=lambda e: e.get('type') == 'cron' and e.get('schedule') == 'daily',
    action=lambda e: 'Tages-Digest erstellt: 3 Emails, 2 Termine, 1 TODO',
    model='haiku'
)
digest_trigger.cron = CronSchedule('0 8 * * *')

# Webhook-Trigger: Email-Benachrichtigung
email_trigger = EventTrigger(
    'email-alert', 'webhook',
    condition=lambda e: e.get('type') == 'email' and 'dringend' in e.get('subject', '').lower(),
    action=lambda e: f'Dringende Email erkannt: "{e.get("subject")}" → Benachrichtigung gesendet',
    model='sonnet'
)

# Health-Check Trigger
health_trigger = EventTrigger(
    'health-check', 'cron',
    condition=lambda e: e.get('type') == 'cron' and e.get('schedule') == 'health',
    action=lambda e: 'Health-Check: Alle Systeme OK',
    model='haiku'
)
health_trigger.cron = CronSchedule('*/30 * * * *')

engine.add(digest_trigger)
engine.add(email_trigger)
engine.add(health_trigger)

# Events simulieren
engine.process_event({'type': 'cron', 'schedule': 'daily'})
engine.process_event({'type': 'email', 'subject': 'DRINGEND: Server down'})
engine.process_event({'type': 'email', 'subject': 'Newsletter Maerz'})  # Kein Match
engine.process_event({'type': 'cron', 'schedule': 'health'})

# Kosten kalkulieren
engine.cost_estimate(days=30)

In [ ]:
# 🎯 ÜBUNG: Erstelle einen eigenen Trigger + Kostenanalyse
#
# Aufgabe:
# 1. Erstelle einen 'weekly-review' Trigger (jeden Freitag 18:30)
# 2. Erstelle einen 'file-watcher' Trigger (reagiert auf Dateiänderungen)
# 3. Fuege beide zur Engine hinzu
# 4. Berechne die monatlichen Kosten fuer alle Trigger

# TODO: Weekly Review Trigger
# weekly = EventTrigger('weekly-review', 'cron', ...)
# weekly.cron = CronSchedule('30 18 * * 5')

# TODO: File-Watcher Trigger
# file_watch = EventTrigger('file-watcher', 'file', ...)

# TODO: Zur Engine hinzufuegen und Kosten berechnen

In [ ]:
# ✅ LÖSUNG

weekly = EventTrigger(
    'weekly-review', 'cron',
    condition=lambda e: e.get('type') == 'cron' and e.get('schedule') == 'weekly',
    action=lambda e: 'Wochenreview: 5 Tasks erledigt, 2 offen, Produktivitaet 78%',
    model='sonnet'  # Komplexere Analyse
)
weekly.cron = CronSchedule('30 18 * * 5')

file_watch = EventTrigger(
    'file-watcher', 'file',
    condition=lambda e: e.get('type') == 'file' and e.get('path', '').endswith('.md'),
    action=lambda e: f'Datei geaendert: {e.get("path")} → Backup erstellt',
    model='haiku'
)
file_watch.execution_count = 10  # Schaetzung: 10 Datei-Aenderungen pro Tag

engine.add(weekly)
engine.add(file_watch)

print('=== Test ===')
engine.process_event({'type': 'cron', 'schedule': 'weekly'})
engine.process_event({'type': 'file', 'path': 'memory/2026-03-24.md'})
engine.process_event({'type': 'file', 'path': 'data.csv'})  # Kein Match (.csv)

engine.cost_estimate(days=30)

## Kosten-Bewusstsein

| Automatisierung | Modell | Geschätzte Kosten/Monat |
|----------------|--------|------------------------|
| Täglicher Digest | Haiku | ~$0.03 |
| Health-Check (30 min) | Haiku | ~$1.44 |
| Weekly Review | Sonnet | ~$0.04 |
| Wöchentl. Deep Research | Sonnet | ~$0.16 |

**Faustregel:** Haiku für Routine, Sonnet für Analyse, Opus nur für komplexe Entscheidungen.

---

### ✅ Self-Check
- [ ] Ich kann einen Cron-Ausdruck lesen und erklären
- [ ] Ich verstehe das Trigger → Condition → Action Pattern
- [ ] Ich kann API-Kosten für Automatisierungen kalkulieren

**→ Weiter zu [Modul 11: Sicherheit & Trust](modul11_security.ipynb)**